In [1]:
import pandas as pd

final_data = pd.read_csv('../data_clean/clean_masterdata.csv',low_memory = False)
print(final_data.head())

   userId  movieId  rating                 genres  \
0       1      296     5.0         Thriller Crime   
1       1      306     3.5  Drama Mystery Romance   
2       1      307     5.0    Drama Music Mystery   
3       1      665     5.0       War Drama Comedy   
4       1      899     3.5   Comedy Music Romance   

                                            overview                title  \
0  A burger-loving hit man, his philosophical par...         Pulp Fiction   
1  Red This is the third film from the trilogy by...    Three Colors: Red   
2  A woman struggles to find a way to live her li...   Three Colors: Blue   
3  Black marketeers Marko (Miki Manojlovic) and B...          Underground   
4  In 1927 Hollywood, Don Lockwood and Lina Lamon...  Singin' in the Rain   

   imdbId  
0  110912  
1  111495  
2  108394  
3  114787  
4   45152  


In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

movies_data = final_data.drop_duplicates(subset=['movieId']).reset_index(drop=True)
#print(f" rows: {len(movies_data)}")

movies_data['mix'] = movies_data['overview'].fillna('') + " " + movies_data['genres'].fillna('') #Fills nan's with '' as pandas automatically fill them after loading csv
tfidf = TfidfVectorizer(stop_words = 'english')
tfidf_matrix = tfidf.fit_transform(movies_data['mix']) # .fit() reads all the overview columns/genres and contain it in a dictionary(dropping stop words) and the .trasnfrom() creates tfidf scores for all the words in that dict
#The matrix space had 40k rows(movies) and the columns are the unique words with the coodrinate having the tfidf score 

cosine_sim = linear_kernel(tfidf_matrix,tfidf_matrix) #calculates the distance between movies (similarity scores)

#print(cosine_sim.shape)


In [21]:
#The search engine

indices = pd.Series(movies_data.index, index = movies_data['title'].str.lower()).drop_duplicates() #Creates a dictory with movie title as keys and their row no.s as value

def get_recommendation(title, cosine_sim=cosine_sim):
    c_title = title.lower().strip()
    if c_title not in indices:
        return f"{title} not in database"

    idx = indices[c_title] #index no. for the title
    if isinstance(idx,pd.Series):
        idx=idx.iloc[0]

    sim_scores = list(enumerate(cosine_sim[idx])) #gives a list of similarity scores corresponding to the row
    sim_scores = sorted(sim_scores, key = lambda x:x[1], reverse = True) #Sorts in reverse of scores

    top_10 = sim_scores[1:11]

    movies_indices = [i[0] for i in top_10] #we just want row numbers corresponding to the top 10 scores (list of row numbers)

    return movies_data['title'].iloc[movies_indices]

print(get_recommendation('the dark knight'))
    
    

    
    

698                                  The Dark Knight Rises
1679                                        Batman Returns
912                                         Batman Forever
33837    LEGO DC Comics Super Heroes: Batman: Be-Leaguered
8999                              Batman: The Killing Joke
25180    Batman Beyond Darwyn Cooke's Batman 75th Anniv...
20809                                    Batman vs Dracula
2524                          Batman: Mask of the Phantasm
14742                                             Ricochet
245                                          Batman Begins
Name: title, dtype: object
